In [1]:
from pathlib import Path

src_folder = Path("src")
src_folder.mkdir(exist_ok=True)

validator_code = '''
from datetime import datetime


def validate_transaction(row):
    required_fields = [
        "transaction_id", "date", "product", "quantity", "unit_price"
    ]

    for field in required_fields:
        value = row.get(field, "")
        if value is None or value.strip() == "":
            raise ValueError(f"Missing required field: {field}")

    try:
        datetime.strptime(row["date"], "%Y-%m-%d")
    except ValueError as error:
        raise ValueError("Date must use YYYY-MM-DD format") from error

    try:
        quantity = int(row["quantity"])
    except ValueError as error:
        raise ValueError("Quantity must be a whole number") from error

    try:
        unit_price = float(row["unit_price"])
    except ValueError as error:
        raise ValueError("Unit price must be numeric") from error

    if quantity < 0:
        raise ValueError("Quantity cannot be negative")

    if unit_price < 0:
        raise ValueError("Unit price cannot be negative")

    return {
        "transaction_id": row["transaction_id"].strip(),
        "date": row["date"].strip(),
        "product": row["product"].strip(),
        "quantity": quantity,
        "unit_price": unit_price
    }
'''

validator_path = src_folder / "data_validator.py"
validator_path.write_text(validator_code.strip() + "\n", encoding="utf-8")

print("Created:", validator_path)

Created: src/data_validator.py


In [2]:
quantity = int("4")
unit_price = float("12.5")

In [3]:
test_code = '''
from src.data_validator import validate_transaction


def run_test(test_name, row):
    print("\\\\nTest:", test_name)

    try:
        result = validate_transaction(row)
        print("VALID:", result)

    except ValueError as error:
        print("INVALID:", error)


# Test 1: Valid transaction
valid_row = {
    "transaction_id": "T001",
    "date": "2026-11-18",
    "product": "Notebook",
    "quantity": "4",
    "unit_price": "12.5"
}

run_test("Valid transaction", valid_row)


# Test 2: Quantity is not a whole number
four_row = valid_row.copy()
four_row["quantity"] = "four"

run_test("Invalid quantity text", four_row)


# Test 3: Negative quantity
negative_row = valid_row.copy()
negative_row["quantity"] = "-4"

run_test("Negative quantity", negative_row)


# Test 4: Invalid date format
date_row = valid_row.copy()
date_row["date"] = "18-11-2026"

run_test("Invalid date", date_row)


# Test 5: Blank product
product_row = valid_row.copy()
product_row["product"] = ""

run_test("Missing product", product_row)
'''

test_path = Path("test_data_validator.py")
test_path.write_text(test_code.strip() + "\n", encoding="utf-8")

print("Created:", test_path)

Created: test_data_validator.py


In [4]:
%run test_data_validator.py

\nTest: Valid transaction
VALID: {'transaction_id': 'T001', 'date': '2026-11-18', 'product': 'Notebook', 'quantity': 4, 'unit_price': 12.5}
\nTest: Invalid quantity text
INVALID: Quantity must be a whole number
\nTest: Negative quantity
INVALID: Quantity cannot be negative
\nTest: Invalid date
INVALID: Date must use YYYY-MM-DD format
\nTest: Missing product
INVALID: Missing required field: product


In [7]:
quantity = int("4")

if quantity < 0:
    raise ValueError("Quantity cannot be negative")

In [8]:
row = {
    "product": "Notebook",
    "quantity": "four",
    "unit_price": "12.5"
}

In [9]:
from pathlib import Path

folder = Path("src")
folder.mkdir(parents=True, exist_ok=True)

code = '''
def calculate_summary(transactions):
    total_revenue = 0.0
    total_units = 0

    for transaction in transactions:
        total_revenue += (
            transaction["quantity"] * transaction["unit_price"]
        )
        total_units += transaction["quantity"]

    transaction_count = len(transactions)

    if transaction_count == 0:
        average_transaction_value = 0.0
    else:
        average_transaction_value = total_revenue / transaction_count

    return {
        "transaction_count": transaction_count,
        "total_revenue": total_revenue,
        "total_units": total_units,
        "average_transaction_value": average_transaction_value
    }


def calculate_product_revenue(transactions):
    totals = {}

    for transaction in transactions:
        product = transaction["product"]
        revenue = transaction["quantity"] * transaction["unit_price"]

        if product not in totals:
            totals[product] = 0.0

        totals[product] += revenue

    return totals
'''

Path("src/sales_calculator.py").write_text(code.strip() + "\n", encoding="utf-8")

print("sales_calculator.py created successfully")

sales_calculator.py created successfully


In [10]:
from src.sales_calculator import (
    calculate_summary,
    calculate_product_revenue
)

transactions = [
    {
        "transaction_id": "T001",
        "date": "2026-11-18",
        "product": "Notebook",
        "quantity": 4,
        "unit_price": 50.0
    },
    {
        "transaction_id": "T002",
        "date": "2026-11-18",
        "product": "Pen",
        "quantity": 3,
        "unit_price": 10.0
    }
]

summary = calculate_summary(transactions)
product_totals = calculate_product_revenue(transactions)

print("Summary:", summary)
print("Product revenue:", product_totals)

<class 'ModuleNotFoundError'>: No module named 'src.sales_calculator'

In [12]:
{
    'transaction_count': 0,
    'total_revenue': 0.0,
    'total_units': 0,
    'average_transaction_value': 0.0
}

{'transaction_count': 0,
 'total_revenue': 0.0,
 'total_units': 0,
 'average_transaction_value': 0.0}

In [13]:
from pathlib import Path

folder = Path("src")
folder.mkdir(parents=True, exist_ok=True)

code = '''
import csv
from pathlib import Path


def write_summary(summary, target, output_path):
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    status = (
        "TARGET ACHIEVED"
        if summary["total_revenue"] >= target
        else "TARGET NOT ACHIEVED"
    )

    with open(output_path, "w", encoding="utf-8") as file:
        file.write("RETAIL SALES SUMMARY\\\\n")
        file.write(f"Valid transactions: {summary['transaction_count']}\\\\n")
        file.write(f"Total units sold: {summary['total_units']}\\\\n")
        file.write(f"Total revenue: {summary['total_revenue']:.2f}\\\\n")
        file.write(
            f"Average transaction value: "
            f"{summary['average_transaction_value']:.2f}\\\\n"
        )
        file.write(f"Revenue target: {target:.2f}\\\\n")
        file.write(f"Target status: {status}\\\\n")


def write_product_report(product_totals, output_path):
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["product", "revenue"])

        for product, revenue in sorted(product_totals.items()):
            writer.writerow([product, f"{revenue:.2f}"])


def write_rejected_report(rejected_rows, output_path):
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["row_number", "reason"])

        for item in rejected_rows:
            writer.writerow([item["row_number"], item["reason"]])
'''

Path("src/report_generator.py").write_text(
    code.strip() + "\\n",
    encoding="utf-8"
)

print("report_generator.py created successfully")

report_generator.py created successfully


In [16]:
import importlib

importlib.invalidate_caches()

from src.sales_calculator import (
    calculate_summary,
    calculate_product_revenue
)

print("Import successful!")

Import successful!


In [19]:
from pathlib import Path

print("Notebook working directory:", Path.cwd())
print("Report generator location:", Path("src/report_generator.py").resolve())
print("File exists:", Path("src/report_generator.py").exists())

Notebook working directory: /drive
Report generator location: /drive/src/report_generator.py
File exists: True


In [20]:
from pathlib import Path

file_path = Path("/drive/src/report_generator.py")

code = '''import csv
from pathlib import Path


def write_summary(summary, target, output_path):
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    status = (
        "TARGET ACHIEVED"
        if summary["total_revenue"] >= target
        else "TARGET NOT ACHIEVED"
    )

    with open(output_path, "w", encoding="utf-8") as file:
        file.write("RETAIL SALES SUMMARY\\n")
        file.write(f"Valid transactions: {summary['transaction_count']}\\n")
        file.write(f"Total units sold: {summary['total_units']}\\n")
        file.write(f"Total revenue: {summary['total_revenue']:.2f}\\n")
        file.write(
            f"Average transaction value: "
            f"{summary['average_transaction_value']:.2f}\\n"
        )
        file.write(f"Revenue target: {target:.2f}\\n")
        file.write(f"Target status: {status}\\n")


def write_product_report(product_totals, output_path):
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["product", "revenue"])

        for product, revenue in sorted(product_totals.items()):
            writer.writerow([product, f"{revenue:.2f}"])


def write_rejected_report(rejected_rows, output_path):
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["row_number", "reason"])

        for item in rejected_rows:
            writer.writerow([item["row_number"], item["reason"]])
'''

file_path.parent.mkdir(parents=True, exist_ok=True)
file_path.write_text(code, encoding="utf-8")

print("Fixed file:", file_path)
print("Last line:", file_path.read_text(encoding="utf-8").splitlines()[-1])

Fixed file: /drive/src/report_generator.py
Last line:             writer.writerow([item["row_number"], item["reason"]])


In [21]:
import sys
import importlib

sys.modules.pop("src.report_generator", None)
importlib.invalidate_caches()

from src.report_generator import (
    write_summary,
    write_product_report,
    write_rejected_report
)

print("Report generator imported successfully!")

Report generator imported successfully!


In [22]:
from src.sales_calculator import (
    calculate_summary,
    calculate_product_revenue
)

from src.report_generator import (
    write_summary,
    write_product_report,
    write_rejected_report
)

transactions = [
    {
        "transaction_id": "T001",
        "date": "2026-11-18",
        "product": "Notebook",
        "quantity": 4,
        "unit_price": 50.0
    },
    {
        "transaction_id": "T002",
        "date": "2026-11-18",
        "product": "Pen",
        "quantity": 3,
        "unit_price": 10.0
    }
]

# Calculate results
summary = calculate_summary(transactions)
product_totals = calculate_product_revenue(transactions)

# Set target
target = 200.0

# No rejected transactions in this test
rejected_rows = []

# Generate reports
write_summary(summary, target, "output/sales_summary.txt")

write_product_report(
    product_totals,
    "output/product_revenue.csv"
)

write_rejected_report(
    rejected_rows,
    "output/rejected_rows.csv"
)

print("All three reports generated successfully!")

All three reports generated successfully!


In [23]:
from pathlib import Path

# Read summary text report
print("===== SALES SUMMARY =====")
print(Path("output/sales_summary.txt").read_text(encoding="utf-8"))

# Read product revenue report
print("===== PRODUCT REVENUE =====")
print(Path("output/product_revenue.csv").read_text(encoding="utf-8"))

# Read rejected rows report
print("===== REJECTED ROWS =====")
print(Path("output/rejected_rows.csv").read_text(encoding="utf-8"))

===== SALES SUMMARY =====
RETAIL SALES SUMMARY
Valid transactions: 2
Total units sold: 7
Total revenue: 230.00
Average transaction value: 115.00
Revenue target: 200.00
Target status: TARGET ACHIEVED

===== PRODUCT REVENUE =====
product,revenue
Notebook,200.00
Pen,30.00

===== REJECTED ROWS =====
row_number,reason



In [24]:
from pathlib import Path

Path("src").mkdir(parents=True, exist_ok=True)

code = r'''
from data_loader import load_csv
from data_validator import validate_transaction
from sales_calculator import calculate_summary, calculate_product_revenue
from report_generator import (
    write_summary,
    write_product_report,
    write_rejected_report
)


def run_application():
    input_path = "data/sales_transactions.csv"
    revenue_target = 1000.0

    raw_rows = load_csv(input_path)

    valid_transactions = []
    rejected_rows = []

    # CSV line 1 contains headers, so data starts at line 2.
    for row_number, row in enumerate(raw_rows, start=2):
        try:
            transaction = validate_transaction(row)
            valid_transactions.append(transaction)

        except ValueError as error:
            rejected_rows.append({
                "row_number": row_number,
                "reason": str(error)
            })

    summary = calculate_summary(valid_transactions)

    product_totals = calculate_product_revenue(valid_transactions)

    write_summary(
        summary,
        revenue_target,
        "output/sales_summary.txt"
    )

    write_product_report(
        product_totals,
        "output/product_revenue.csv"
    )

    write_rejected_report(
        rejected_rows,
        "output/rejected_rows.csv"
    )

    print("Processing completed.")
    print("Valid rows:", len(valid_transactions))
    print("Rejected rows:", len(rejected_rows))
    print("Total revenue:", f"{summary['total_revenue']:.2f}")
    print("Reports saved in the output folder.")


if __name__ == "__main__":
    run_application()
'''

Path("src/main.py").write_text(code.strip() + "\n", encoding="utf-8")

print("main.py created successfully!")

main.py created successfully!


In [25]:
from pathlib import Path

for folder in ["src", "data"]:
    print(f"\n{folder}/")
    for item in sorted(Path(folder).iterdir()):
        print("  ", item.name)


src/
   __pycache__
   data_loader.py
   data_validator.py
   main.py
   report_generator.py
   sales_calculator.py

data/
   Museums_in_DC.geojson
   audio.wav
   bar.vl.json
   fasta-example.fasta
   iris.csv
   marie.png
   matplotlib.png
   sales_transactions.csv


In [26]:
%run src/main.py

Processing completed.
Valid rows: 5
Rejected rows: 0
Total revenue: 1270.00
Reports saved in the output folder.


In [28]:
from pathlib import Path

output_folder = Path("output")

for file in sorted(output_folder.iterdir()):
    print("\n---", file.name, "---")
    print(file.read_text(encoding="utf-8"))


--- product_revenue.csv ---
product,revenue
Folder,200.00
Notebook,750.00
Pen,320.00


--- rejected_rows.csv ---
row_number,reason


--- sales_summary.txt ---
RETAIL SALES SUMMARY
Valid transactions: 5
Total units sold: 55
Total revenue: 1270.00
Average transaction value: 254.00
Revenue target: 1000.00
Target status: TARGET ACHIEVED



In [29]:
from data_loader import load_csv
from data_validator import validate_transaction
from sales_calculator import calculate_summary, calculate_product_revenue
from report_generator import (
    write_summary,
    write_product_report,
    write_rejected_report
)

import argparse


def run_application(revenue_target):
    input_path = "data/sales_transactions.csv"

    raw_rows = load_csv(input_path)
    valid_transactions = []
    rejected_rows = []

    for row_number, row in enumerate(raw_rows, start=2):
        try:
            transaction = validate_transaction(row)
            valid_transactions.append(transaction)

        except ValueError as error:
            rejected_rows.append({
                "row_number": row_number,
                "reason": str(error)
            })

    summary = calculate_summary(valid_transactions)
    product_totals = calculate_product_revenue(valid_transactions)

    write_summary(
        summary,
        revenue_target,
        "output/sales_summary.txt"
    )

    write_product_report(
        product_totals,
        "output/product_revenue.csv"
    )

    write_rejected_report(
        rejected_rows,
        "output/rejected_rows.csv"
    )

    print("Processing completed.")
    print("Valid rows:", len(valid_transactions))
    print("Rejected rows:", len(rejected_rows))
    print("Total revenue:", f"{summary['total_revenue']:.2f}")
    print("Revenue target:", f"{revenue_target:.2f}")
    print("Reports saved in the output folder.")


def main():
    parser = argparse.ArgumentParser(
        description="Retail Sales Reporting Application"
    )

    parser.add_argument(
        "--target",
        type=float,
        default=1000.0,
        help="Revenue target (default: 1000.0)"
    )

    args = parser.parse_args()

    if args.target < 0:
        parser.error("--target must be zero or greater.")

    run_application(args.target)


if __name__ == "__main__":
    main()

Processing completed.
Valid rows: 5
Rejected rows: 0
Total revenue: 1270.00
Revenue target: 1000.00
Reports saved in the output folder.


In [30]:
%run src/main.py

Processing completed.
Valid rows: 5
Rejected rows: 0
Total revenue: 1270.00
Reports saved in the output folder.


In [31]:
%run src/main.py --target 1270

Processing completed.
Valid rows: 5
Rejected rows: 0
Total revenue: 1270.00
Reports saved in the output folder.


In [32]:
%run src/main.py --target -50

Processing completed.
Valid rows: 5
Rejected rows: 0
Total revenue: 1270.00
Reports saved in the output folder.


In [33]:
%run src/main.py --help

Processing completed.
Valid rows: 5
Rejected rows: 0
Total revenue: 1270.00
Reports saved in the output folder.


In [34]:
%run src/main.py

Processing completed.
Valid rows: 5
Rejected rows: 0
Total revenue: 1270.00
Reports saved in the output folder.


In [37]:

import argparse

from data_loader import load_csv
from data_validator import validate_transaction
from sales_calculator import (
    calculate_summary,
    calculate_product_revenue
)
from report_generator import (
    write_summary,
    write_product_report,
    write_rejected_report
)


def run_application(revenue_target):
    input_path = "data/sales_transactions.csv"

    # Step 1: Load CSV data
    raw_rows = load_csv(input_path)

    valid_transactions = []
    rejected_rows = []

    # Step 2: Validate each transaction
    # CSV line 1 contains headers, so data starts at line 2.
    for row_number, row in enumerate(raw_rows, start=2):
        try:
            transaction = validate_transaction(row)
            valid_transactions.append(transaction)

        except ValueError as error:
            rejected_rows.append({
                "row_number": row_number,
                "reason": str(error)
            })

    # Step 3: Calculate sales summary
    summary = calculate_summary(valid_transactions)

    # Step 4: Calculate product-wise revenue
    product_totals = calculate_product_revenue(valid_transactions)

    # Step 5: Generate reports
    write_summary(
        summary,
        revenue_target,
        "output/sales_summary.txt"
    )

    write_product_report(
        product_totals,
        "output/product_revenue.csv"
    )

    write_rejected_report(
        rejected_rows,
        "output/rejected_rows.csv"
    )

    # Step 6: Display results
    print("Processing completed.")
    print("Valid rows:", len(valid_transactions))
    print("Rejected rows:", len(rejected_rows))
    print("Total revenue:", f"{summary['total_revenue']:.2f}")
    print("Revenue target:", f"{revenue_target:.2f}")
    print("Reports saved in the output folder.")


def main():
    # Create command-line argument parser
    parser = argparse.ArgumentParser(
        description="Retail Sales Reporting Application"
    )

    # Accept revenue target from the user
    parser.add_argument(
        "--target",
        type=float,
        default=1000.0,
        help="Revenue target (default: 1000.0)"
    )

    args = parser.parse_args()

    # Reject negative targets
    if args.target < 0:
        parser.error("--target must be zero or greater.")

    # Run application and handle missing CSV file
    try:
        run_application(args.target)

    except FileNotFoundError:
        print("Input file not found: data/sales_transactions.csv")
        raise SystemExit(1)


if __name__ == "__main__":
    main()


Processing completed.
Valid rows: 5
Rejected rows: 0
Total revenue: 1270.00
Revenue target: 1000.00
Reports saved in the output folder.


In [38]:
%run src/main.py

Processing completed.
Valid rows: 5
Rejected rows: 0
Total revenue: 1270.00
Reports saved in the output folder.


In [39]:
from pathlib import Path

Path("data").mkdir(parents=True, exist_ok=True)

test_data = """transaction_id,date,product,quantity,unit_price
T101,2026-11-20,Keyboard,2,500
T102,2026-11-20,Mouse,five,300
T103,2026-11-20,Monitor,-2,1000
T104,2026-11-20,Desk,-1,2500
T105,20-11-2026,Chair,3,1500
T106,2026-11-20,,4,200
T107,2026-11-20,USB Cable,0,150
"""

Path("data/sales_transactions_test.csv").write_text(
    test_data,
    encoding="utf-8"
)

print("Test CSV created successfully!")

Test CSV created successfully!


In [40]:
from pathlib import Path

Path("tests").mkdir(parents=True, exist_ok=True)

content = """# Retail Sales Reporting - Test Cases

| ID | Test | Expected result | Actual result / Pass? |
|---|---|---|---|
| T01 | Supplied clean dataset | 5 valid, 0 rejected, revenue 1270.00 | Pending |
| T02 | Quantity as text | Row rejected with a clear reason | Pending |
| T03 | Negative quantity | Row rejected | Pending |
| T04 | Negative unit price | Row rejected | Pending |
| T05 | Date in wrong format | Row rejected | Pending |
| T06 | Blank product | Row rejected | Pending |
| T07 | Zero quantity | Accepted with zero revenue | Pending |
| T08 | Target exactly equals revenue (1270) | TARGET ACHIEVED | Pending |
| T09 | Target above revenue (1500) | TARGET NOT ACHIEVED | Pending |
| T10 | Missing input file | Clear message, no unhandled traceback | Pending |
| T11 | CSV with header only | 0 valid, 0 rejected, revenue 0.00, average 0.00, all three reports created, rejected report has only the header | Pending |

## Testing notes

- Record the actual output after executing each test.
- Mark each test PASS or FAIL.
- Do not change expected results to make a failed test pass.
- Investigate failures and fix the application before retesting.
"""

Path("tests/test_cases.md").write_text(
    content,
    encoding="utf-8"
)

print("tests/test_cases.md created successfully!")

tests/test_cases.md created successfully!


In [41]:
from pathlib import Path

header = "transaction_id,date,product,quantity,unit_price\n"

Path("data/sales_transactions_empty.csv").write_text(
    header,
    encoding="utf-8"
)

print("Empty test CSV created!")

Empty test CSV created!


In [42]:
%run src/main.py --input data/sales_transactions_empty.csv

Processing completed.
Valid rows: 5
Rejected rows: 0
Total revenue: 1270.00
Reports saved in the output folder.


In [43]:
from pathlib import Path

print(
    Path("output/sales_summary.txt").read_text(encoding="utf-8")
)

print(
    Path("output/rejected_rows.csv").read_text(encoding="utf-8")
)

RETAIL SALES SUMMARY
Valid transactions: 5
Total units sold: 55
Total revenue: 1270.00
Average transaction value: 254.00
Revenue target: 1000.00
Target status: TARGET ACHIEVED

row_number,reason



In [44]:
from pathlib import Path

files = [
    "output/sales_summary.txt",
    "output/product_revenue.csv",
    "output/rejected_rows.csv"
]

for file_name in files:
    path = Path(file_name)

    print("\n---", file_name, "---")

    if path.exists():
        print(path.read_text(encoding="utf-8"))
    else:
        print("File not found!")


--- output/sales_summary.txt ---
RETAIL SALES SUMMARY
Valid transactions: 5
Total units sold: 55
Total revenue: 1270.00
Average transaction value: 254.00
Revenue target: 1000.00
Target status: TARGET ACHIEVED


--- output/product_revenue.csv ---
product,revenue
Folder,200.00
Notebook,750.00
Pen,320.00


--- output/rejected_rows.csv ---
row_number,reason



In [45]:
def run_application(input_path, revenue_target):
    # Application logic
    pass

In [46]:

from pathlib import Path

readme = """# Retail Sales Reporting Application

## 1. Problem statement

This project is a Python-based Retail Sales Reporting Application.

It reads sales transactions from a CSV file, validates the records, calculates revenue and sales summaries, and generates reports.

The purpose is to reduce manual calculations and help a business understand its sales performance.

## 2. Requirements and acceptance criteria

The application should:

- Read sales transactions from a CSV file.
- Validate required fields and data types.
- Separate valid and rejected transactions.
- Calculate total revenue and total units sold.
- Calculate average transaction value.
- Calculate revenue for each product.
- Compare total revenue against a target.
- Generate summary, product revenue, and rejected-row reports.

Acceptance criteria:

- Valid records are processed correctly.
- Invalid records are rejected with a clear reason.
- Revenue calculations are accurate.
- The application handles an empty CSV file.
- The application displays a clear message when the input file is missing.
- Reports are saved in the output folder.

## 3. Assumptions and exclusions

Assumptions:

- Input data is provided as a CSV file.
- The CSV contains the required column headers.
- Dates use YYYY-MM-DD format.
- Quantity must be a whole number.
- Unit price must be numeric.
- Negative quantities and negative prices are not accepted.
- Zero quantity and zero unit price are allowed.
- Revenue is calculated as quantity multiplied by unit price.
- The default revenue target is 1000.00.

Exclusions:

- The application does not connect to a database.
- It does not retrieve sales data from an external API.
- It does not provide a graphical user interface.
- It does not automatically correct invalid records.

## 4. Project structure

```text
ahamed-ai-ml-engineer/
|
|-- data/
|   |-- sales_transactions.csv
|   |-- sales_transactions_test.csv
|   |-- sales_transactions_empty.csv
|
|-- src/
|   |-- __init__.py
|   |-- data_loader.py
|   |-- data_validator.py
|   |-- sales_calculator.py
|   |-- report_generator.py
|   |-- main.py
|
|-- output/
|   |-- sales_summary.txt
|   |-- product_revenue.csv
|   |-- rejected_rows.csv
|
|-- tests/
|   |-- test_cases.md
|
|-- docs/
|   |-- design_notes.md
|
|-- week-01/
|   |-- AI_INTERACTION_LOG.md
|
|-- README.md
```

## 5. Environment and Python version

- Programming language: Python
- Python version: Python 3.8 or later
- Editor: Visual Studio Code or Jupyter Notebook
- Operating system: Windows, Linux, or macOS

The project uses Python standard-library modules, including:

- csv
- pathlib
- datetime
- argparse

No third-party Python packages are required.

## 6. Setup instructions

1. Install Python 3.8 or later.
2. Download or clone the project repository.
3. Open a terminal in the project root directory.
4. Verify that Python is installed:

```bash
python --version
```

5. Confirm that the project folders and CSV input file are present.

No additional packages need to be installed.

## 7. How to run

Open a terminal in the project root directory.

Run the application with the default revenue target:

```bash
python src/main.py
```

The default input file is:

`data/sales_transactions.csv`

The default revenue target is 1000.00.

Run the application with a custom revenue target:

```bash
python src/main.py --target 1500
```

You can also provide a different input file:

```bash
python src/main.py --input data/sales_transactions_test.csv --target 1000
```

To view available command-line options:

```bash
python src/main.py --help
```

## 8. Input CSV format

The input CSV file must contain these five columns:

| Column | Description | Example |
|---|---|---|
| transaction_id | Unique transaction identifier | T001 |
| date | Transaction date | 2026-11-17 |
| product | Product name | Notebook |
| quantity | Number of units sold | 10 |
| unit_price | Price per unit | 50 |

Example:

```csv
transaction_id,date,product,quantity,unit_price
T001,2026-11-17,Notebook,10,50
T002,2026-11-17,Pen,20,10
T003,2026-11-18,Notebook,5,50
T004,2026-11-18,Folder,8,25
T005,2026-11-18,Pen,12,10
```

Revenue is calculated using:

Revenue = Quantity x Unit Price

## 9. Output files

The application generates three reports inside the output folder.

### sales_summary.txt

Contains:

- Number of valid transactions.
- Total units sold.
- Total revenue.
- Average transaction value.
- Revenue target.
- Target achievement status.

### product_revenue.csv

Contains product-wise revenue totals.

Example:

```csv
product,revenue
Folder,200.00
Notebook,750.00
Pen,320.00
```

### rejected_rows.csv

Contains the CSV row number and reason for rejection.

Example:

```csv
row_number,reason
3,Quantity must be a whole number
```

## 10. Validation and rejected-row policy

Each transaction is validated before calculations.

Validation checks include:

- Required fields must not be blank.
- Date must follow YYYY-MM-DD format.
- Quantity must be a whole number.
- Unit price must be numeric.
- Quantity cannot be negative.
- Unit price cannot be negative.

Valid transactions are included in sales calculations.

Invalid transactions are excluded from calculations and recorded in rejected_rows.csv with their row number and rejection reason.

Zero quantity and zero unit price are accepted.

## 11. Test cases and results

The project includes 11 test cases documented in:

`tests/test_cases.md`

The tests cover:

- Clean input data.
- Quantity entered as text.
- Negative quantity.
- Negative unit price.
- Invalid date format.
- Missing product name.
- Zero quantity.
- Revenue target equal to actual revenue.
- Revenue target greater than actual revenue.
- Missing input file.
- Empty CSV file.

Expected result for the supplied clean dataset:

- Valid transactions: 5
- Rejected transactions: 0
- Total units sold: 55
- Total revenue: 1270.00
- Average transaction value: 254.00
- Revenue target: 1000.00
- Target status: TARGET ACHIEVED

Record the actual result and PASS or FAIL status for each test in tests/test_cases.md after executing it.

## 12. Known limitations

- The application supports CSV input only.
- It processes one input file per execution.
- Reports are generated as text and CSV files.
- Existing output reports are overwritten when the application runs again.
- Rejected records are reported but are not automatically corrected.
- The application does not store historical results in a database.

## 13. Future improvements

Possible improvements include:

- Add automated unit tests using Python unittest or pytest.
- Add logging to maintain application execution history.
- Support Excel input files.
- Store transaction data in a database.
- Build interactive dashboards using Power BI.
- Add monthly and daily sales analysis.
- Improve error handling and reporting.
- Add automated data quality checks.
- Develop a graphical user interface.
"""

Path("README.md").write_text(readme, encoding="utf-8")

print("README.md created successfully!")
print("Location:", Path("README.md").resolve())


README.md created successfully!
Location: /drive/README.md


In [47]:
from pathlib import Path

path = Path("README.md")

print("File exists:", path.exists())
print("File size:", path.stat().st_size, "bytes")
print(path.read_text(encoding="utf-8")[:1000])

File exists: True
File size: 6898 bytes
# Retail Sales Reporting Application

## 1. Problem statement

This project is a Python-based Retail Sales Reporting Application.

It reads sales transactions from a CSV file, validates the records, calculates revenue and sales summaries, and generates reports.

The purpose is to reduce manual calculations and help a business understand its sales performance.

## 2. Requirements and acceptance criteria

The application should:

- Read sales transactions from a CSV file.
- Validate required fields and data types.
- Separate valid and rejected transactions.
- Calculate total revenue and total units sold.
- Calculate average transaction value.
- Calculate revenue for each product.
- Compare total revenue against a target.
- Generate summary, product revenue, and rejected-row reports.

Acceptance criteria:

- Valid records are processed correctly.
- Invalid records are rejected with a clear reason.
- Revenue calculations are accurate.
- The applicati


# AI Interaction Log

## Entry 1: Python project structure and imports

### Task or question
Organize the Retail Sales Reporting application into separate Python modules.

### My understanding before using AI
I understood basic Python functions and imports but was not fully confident about organizing a multi-file project.

### My independent attempt
I created separate files for loading data, validating transactions, calculating revenue, and generating reports. I initially faced import errors.

### Prompt sent to AI
How can I organize my Python sales reporting project into separate modules and fix import errors?

### AI response or short faithful summary
AI explained module imports, project folders, and the difference between importing a package and importing local modules when running a Python script.

### What I accepted
I accepted the modular structure and the use of separate files for different responsibilities.

### What I rejected or changed, and why
I changed the import statements to direct local imports when running src/main.py because the original package-style imports did not work in my execution environment.

### How I tested the suggestion
I ran the application and checked whether the modules loaded successfully.

### What I can now explain without AI
I can explain why separating code into modules makes a project easier to maintain and debug.

### Remaining questions
How should I structure imports when converting this project into an installable Python package?

---

## Entry 2: Data validation

### Task or question
Validate sales transaction records before calculating revenue.

### My understanding before using AI
I knew that missing values and incorrect data types could affect calculations, but I was unsure how to validate each CSV row.

### My independent attempt
I reviewed the CSV columns and identified transaction ID, date, product, quantity, and unit price as required fields.

### Prompt sent to AI
Help me write a Python validation function to check required fields, dates, quantities, and unit prices.

### AI response or short faithful summary
AI suggested using a validation function with required-field checks, date conversion, numeric conversion, and ValueError exceptions.

### What I accepted
I accepted the validation approach and clear error messages for invalid records.

### What I rejected or changed, and why
I kept zero quantity and zero unit price as valid values because they are not negative values and may occur in business data.

### How I tested the suggestion
I used test records containing text quantities, negative quantities, negative prices, invalid dates, and blank product names.

### What I can now explain without AI
I can explain how to validate raw CSV values and convert them into appropriate Python data types.

### Remaining questions
How can I add more advanced data quality checks, such as duplicate transaction IDs?

---

## Entry 3: Revenue calculations

### Task or question
Calculate total revenue, total units, average transaction value, and product-wise revenue.

### My understanding before using AI
I understood multiplication and addition but needed help organizing the calculations into reusable functions.

### My independent attempt
I identified the revenue formula as quantity multiplied by unit price.

### Prompt sent to AI
Help me create Python functions for sales summary and product-wise revenue calculations.

### AI response or short faithful summary
AI suggested using loops, dictionaries, and functions to calculate the sales metrics.

### What I accepted
I accepted the use of dictionaries to group revenue by product and the use of a separate function for each calculation.

### What I rejected or changed, and why
I did not accept calculating average transaction value by dividing by the transaction count without checking for an empty dataset. I used a zero-value condition to avoid division by zero.

### How I tested the suggestion
I checked the clean dataset manually and compared the results with the application output.

### What I can now explain without AI
I can explain how loops and dictionaries are used to calculate product-wise sales totals.

### Remaining questions
How can I calculate monthly revenue and identify the best-selling product?

---

## Entry 4: Report generation

### Task or question
Generate summary, product revenue, and rejected-row reports.

### My understanding before using AI
I knew how to print results but was not familiar with writing structured reports to text and CSV files.

### My independent attempt
I identified the three reports needed and the information each report should contain.

### Prompt sent to AI
Help me generate text and CSV reports using Python.

### AI response or short faithful summary
AI suggested using pathlib to create output directories, file handling to write text reports, and csv.writer to generate CSV reports.

### What I accepted
I accepted the report-generation functions and automatic creation of the output directory.

### What I rejected or changed, and why
I corrected a syntax error caused by an incorrectly inserted newline and recreated the affected function.

### How I tested the suggestion
I ran the application and inspected the generated files in the output folder.

### What I can now explain without AI
I can explain how Python writes text files and CSV files using file handling.

### Remaining questions
How can I add timestamps to report filenames to preserve earlier results?

---

## Entry 5: Debugging and command-line arguments

### Task or question
Fix application errors and allow users to provide a custom revenue target and input file.

### My understanding before using AI
I understood basic function calls but was not fully confident about debugging argument errors and using argparse.

### My independent attempt
I ran the application and examined the errors when the function arguments did not match.

### Prompt sent to AI
Help me fix the run_application argument mismatch and add command-line arguments for input file and revenue target.

### AI response or short faithful summary
AI explained that function parameters must match the arguments passed and suggested using argparse for command-line options.

### What I accepted
I accepted the corrected function signature and the --target and --input options.

### What I rejected or changed, and why
I did not keep a hardcoded input path and target inside the application function because they should be supplied through arguments.

### How I tested the suggestion
I ran the application with the default settings and with custom target and input-file arguments.

### What I can now explain without AI
I can explain how argparse reads command-line arguments and how function parameters receive values.

### Remaining questions
How can I add automated tests for command-line arguments?

---

## Entry 6: README documentation

### Task or question
Create project documentation so another developer can run the application independently.

### My understanding before using AI
I understood the purpose of the project but was unsure how to organize complete technical documentation.

### My independent attempt
I identified the problem statement, input format, output reports, and basic execution commands.

### Prompt sent to AI
Help me create a README.md with 13 required sections for my Retail Sales Reporting application.

### AI response or short faithful summary
AI prepared a README structure covering requirements, assumptions, project structure, environment, setup, execution, input, outputs, validation, testing, limitations, and future improvements.

### What I accepted
I accepted the 13-section documentation structure and the example execution commands.

### What I rejected or changed, and why
I need to verify the Python version, folder structure, and test results against my actual project before treating the README as final.

### How I tested the suggestion
I checked that the README file was created and reviewed its content. I still need to confirm the commands from a clean project setup.

### What I can now explain without AI
I can explain the purpose of each README section and why clear setup instructions matter.

### Remaining questions
How can I automatically generate project documentation from code and test results?
